# Detect personality on the test split directory

Runs the full prediction pipeline from `app_prediction/personality_detector.py` over the
videos in the dataset test directory and logs one CSV per run.

Pipeline per video (`PersonalityPredictor.predict`):
1. Audio  : decode to PCM via PyAV -> VGGish embeddings -> (15, 128)
2. Visual : sample 30 frames -> VGG-Face embeddings (DeepFace) -> (30, 4096)
3. Score  : z-score normalize with training stats, predict with both BiLSTM models,
   then late-fuse (average) into a single OCEAN result

Output: `results/test_detections/detection_<YYYYmmdd_HHMMSS>.csv`, one row per video with
audio/visual/fused scores for the 5 OCEAN traits, processing time, and status.

Slow (~1-2 min per video). Use `LIMIT` in the config cell to cap how many videos run.

In [1]:
import csv
import glob
import os
import sys
import time

app_dir = "/mnt/4A3ED7573ED73AA1/aa-kuliah/skripsi/app"
project_root = os.path.dirname(app_dir)
sys.path.insert(0, os.path.join(app_dir, "app_prediction"))

from app_prediction.personality_detector import OCEAN, warmup

print("OCEAN traits:", OCEAN)

OCEAN traits: ['extraversion', 'neuroticism', 'agreeableness', 'conscientiousness', 'openness']


## Config

- `dataset_test_dir` : test split directory inside the dataset folder (flat `.mp4` files)
- `results_dir` : separate folder for detection logs, one CSV per run
- `LIMIT` : how many videos to detect. `10` = first 10 sorted videos. `None` or `0` = unlimited.
  Re-run this cell after changing `LIMIT`.

In [2]:
dataset_test_dir = os.path.join(project_root, "first-impressions", "test")
results_dir = os.path.join(app_dir, "results", "test_detections")
video_extensions = (".mp4", ".avi", ".mov", ".mkv", ".webm", ".m4v")

LIMIT = 10

CSV_COLUMNS = (
    ["timestamp", "video_id", "video_file", "process_time"]
    + [f"{modality}_{trait}" for modality in ("audio", "visual", "fused") for trait in OCEAN]
    + ["status", "reason"]
)

print("test dir   :", dataset_test_dir)
print("results dir:", results_dir)
print("limit      :", LIMIT if LIMIT else "unlimited")

test dir   : /mnt/4A3ED7573ED73AA1/aa-kuliah/skripsi/first-impressions/test
results dir: /mnt/4A3ED7573ED73AA1/aa-kuliah/skripsi/app/results/test_detections
limit      : 10


## Run detection

Loads the models once, then detects each selected video. A failing video is logged with
`status=ERROR` and the run continues.

In [3]:
def now_ts():
    return time.strftime("%Y-%m-%dT%H:%M:%S")


def get_video_files(directory):
    """Return sorted list of video paths in directory (non-recursive)."""
    paths = []
    for ext in video_extensions:
        paths.extend(glob.glob(os.path.join(directory, f"*{ext}")))
    return sorted(set(paths))


def video_id_of(video_path):
    return os.path.splitext(os.path.basename(video_path))[0]


def detect_video(predictor, video_path):
    """Return (scores, elapsed_seconds). Raises on failure."""
    start = time.perf_counter()
    scores = predictor.predict(video_path)
    return scores, time.perf_counter() - start


def base_row(video_path, elapsed):
    return {
        "timestamp": now_ts(),
        "video_id": video_id_of(video_path),
        "video_file": os.path.basename(video_path),
        "process_time": f"{elapsed:.3f}",
        "status": "OK",
        "reason": "",
    }


videos = get_video_files(dataset_test_dir)
found = len(videos)
if LIMIT:
    videos = videos[:LIMIT]
print(f"{found} videos in test dir, detecting {len(videos)}")

predictor = warmup()

rows = []
for i, video_path in enumerate(videos, start=1):
    start = time.perf_counter()
    print(f"[{i}/{len(videos)}] {video_id_of(video_path)}")
    try:
        scores, elapsed = detect_video(predictor, video_path)
        row = base_row(video_path, elapsed)
        for modality, values in scores.items():
            for trait, score in values.items():
                row[f"{modality}_{trait}"] = f"{score:.6f}"
        print(f"    ok in {elapsed:.1f}s")
    except Exception as e:
        row = base_row(video_path, time.perf_counter() - start)
        row["status"] = "ERROR"
        row["reason"] = f"{type(e).__name__}: {e}"[:300]
        print(f"    error: {row['reason']}")
    rows.append(row)

print(f"\ndetected {len(rows)} videos")

2000 videos in test dir, detecting 10


I0000 00:00:1791271054.703333  437993 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791271057.277080  437993 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


loading audio BiLSTM model ...
loading visual BiLSTM model ...


/mnt/4A3ED7573ED73AA1/aa-kuliah/skripsi/app/venv/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'loss_scale_optimizer', because it has 26 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(store)
/mnt/4A3ED7573ED73AA1/aa-kuliah/skripsi/app/venv/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 22 variables whereas the saved optimizer has 0 variables. 
  saveable.load_own_variables(store)


[1/10] --Ymqszjv54.000
extracting audio features ...
extracting visual features ...
26-10-06 14:17:50 - ⚠️ 
 ⚠️ DEPRECATION WARNING:
 Running 'pip install deepface' alone will no longer be sufficient and will
 NOT install a default backend in an upcoming major release.

 Currently, TensorFlow is included by default, but this behavior will be deprecated.
 Please explicitly specify your preferred backend engine when installing:

   -> pip install deepface[tensorflow]
   -> pip install deepface[pytorch]

 Otherwise, you will encounter 'module not found' errors.



E0000 00:00:1791271080.462884  437993 util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


    ok in 20.7s
[2/10] -10-QQDO_ME.001
extracting audio features ...
extracting visual features ...
    ok in 3.5s
[3/10] -10-QQDO_ME.002
extracting audio features ...
extracting visual features ...
    ok in 3.5s
[4/10] -10-QQDO_ME.005
extracting audio features ...
extracting visual features ...
    ok in 3.4s
[5/10] -4J4xkfN5cI.000
extracting audio features ...
extracting visual features ...
    ok in 3.4s
[6/10] -4J4xkfN5cI.002
extracting audio features ...
extracting visual features ...
    ok in 8.5s
[7/10] -6otZ7M-Mro.002
extracting audio features ...
extracting visual features ...
    ok in 3.1s
[8/10] -6otZ7M-Mro.004
extracting audio features ...
extracting visual features ...
    ok in 3.2s
[9/10] -8asrRvfJWA.002
extracting audio features ...
extracting visual features ...
    ok in 3.5s
[10/10] -DOqN0d8KHw.004
extracting audio features ...
extracting visual features ...
    ok in 3.3s

detected 10 videos


## Save log

Writes this run to its own timestamped CSV in `results/test_detections/`.

In [4]:
os.makedirs(results_dir, exist_ok=True)
csv_path = os.path.join(results_dir, f"detection_{time.strftime('%Y%m%d_%H%M%S')}.csv")

with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
    writer.writeheader()
    writer.writerows(rows)

ok_rows = [row for row in rows if row["status"] == "OK"]
print("csv        :", csv_path)
print("rows       :", len(rows))
print("errors     :", len(rows) - len(ok_rows))
if ok_rows:
    times = [float(row["process_time"]) for row in ok_rows]
    print(f"avg time   : {sum(times) / len(times):.1f}s per video")

csv        : /mnt/4A3ED7573ED73AA1/aa-kuliah/skripsi/app/results/test_detections/detection_20261006_141844.csv
rows       : 10
errors     : 0
avg time   : 5.6s per video


In [5]:
import pandas as pd

df = pd.DataFrame(rows, columns=CSV_COLUMNS)
fused = [f"fused_{trait}" for trait in OCEAN]
display(df[["video_id", "process_time", *fused, "status"]])

,video_id,process_time,fused_extraversion,fused_neuroticism,fused_agreeableness,fused_conscientiousness,fused_openness,status
0,--Ymqszjv54.000,20.672,0.364205,0.397681,0.492956,0.423214,0.491734,OK
1,-10-QQDO_ME.001,3.460,0.496144,0.529889,0.511370,0.549467,0.616803,OK
2,-10-QQDO_ME.002,3.456,0.562596,0.595658,0.569327,0.605039,0.677234,OK
3,-10-QQDO_ME.005,3.450,0.572143,0.579749,0.554374,0.583784,0.647392,OK
4,-4J4xkfN5cI.000,3.439,0.461399,0.466656,0.506313,0.452409,0.544914,OK
5,-4J4xkfN5cI.002,8.470,0.488246,0.494183,0.521963,0.481994,0.584770,OK
6,-6otZ7M-Mro.002,3.144,0.497392,0.559191,0.666123,0.643090,0.637169,OK
7,-6otZ7M-Mro.004,3.199,0.460065,0.561353,0.628057,0.628144,0.608961,OK
8,-8asrRvfJWA.002,3.506,0.527355,0.556684,0.546916,0.577451,0.617407,OK
9,-DOqN0d8KHw.004,3.335,0.476211,0.468744,0.523503,0.466580,0.553937,OK
